<a href="https://colab.research.google.com/github/frank-morales2020/AST/blob/main/deepseek_agentic_gguf.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
!ls -ltha /content/drive/MyDrive/topogpt/

In [ ]:
!ls -ltha /content/drive/MyDrive/GGUF_Models/

In [3]:
!ls -ltha /content/drive/MyDrive/GGUF_Models/deepseek*

-rw------- 1 root root 11G Nov 12 06:55 /content/drive/MyDrive/GGUF_Models/deepseek_moe_16b_topocbp_q4_k_m.gguf
-rw------- 1 root root 70G Nov  9 06:01 /content/drive/MyDrive/GGUF_Models/deepseek-r1-distill-llama-70b-topo-governed-q8_0.gguf


In [4]:
!pip install llama-cpp-python \
  --extra-index-url https://abetlen.github.io/llama-cpp-python/whl/cu130 \
  --no-cache-dir

Looking in indexes: https://pypi.org/simple, https://abetlen.github.io/llama-cpp-python/whl/cu130
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 782.8/782.8 MB 227.1 MB/s  0:00:03
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2/2 [llama-cpp-python]


In [1]:
!nvidia-smi

Thu Nov 12 13:20:55 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA RTX PRO 6000 Blac...    Off |   00000000:05:00.0 Off |                    0 |
| N/A   33C    P0             62W /  600W |       0MiB /  97887MiB |      0%      Default |
|                                         |                        |             Disabled |
+-----------------------------------------+-----

## case1

In [2]:
from llama_cpp import Llama

class DualModelSovereignOrchestrator:
    """
    SOMALA Dual-Model Agentic Orchestrator
    Full production-ready implementation using Google Drive GGUF paths,
    full GPU offloading (-1), and raw prompt formatting.
    """
    def __init__(self, moe_gguf_path: str, r1_70b_gguf_path: str):
        print("[*] Initializing SOMALA Dual-Model Agentic Orchestrator from Google Drive...")

        # Initialize Model 1: DeepSeek-MoE-16B (Fast Router / Worker)
        print(f"[*] Loading MoE from: {moe_gguf_path}")
        self.moe_engine = Llama(
            model_path=moe_gguf_path,
            n_ctx=4096,
            n_threads=4,
            n_gpu_layers=-1,
            verbose=False
        )

        # Initialize Model 2: DeepSeek-R1-Distill-Llama-70B (Deep Reasoner)
        print(f"[*] Loading R1-70B from: {r1_70b_gguf_path}")
        self.r1_engine = Llama(
            model_path=r1_70b_gguf_path,
            n_ctx=8192,
            n_threads=8,
            n_gpu_layers=-1,
            verbose=False
        )
        print("[+] Dual-Model Bare-Metal Engine Active under Topo-CBP Governance.")

    def run_agentic_workflow(self, initial_query: str) -> dict:
        print(f"\n[Step 1] Dispatching query to DeepSeek-MoE-16B for task decomposition...")

        moe_prompt = (
            f"System: You are the MoE routing and task decomposition worker governed by Topo-CBP. "
            f"Deconstruct incoming queries into concise analytical subtasks.\n"
            f"User: Deconstruct this query into core analytical subtasks: {initial_query}\n"
            f"Assistant:"
        )

        moe_response = self.moe_engine(
            moe_prompt,
            max_tokens=512,
            temperature=0.3,
            stop=["User:", "System:", "\n\n\n"]
        )
        subtasks = moe_response["choices"][0]["text"].strip()
        print(f"[+] MoE Decomposition complete:\n{subtasks}")

        print(f"\n[Step 2] Passing subtasks to DeepSeek-R1-Distill-Llama-70B for zero-drift reasoning...")

        r1_prompt = (
            f"System: You are the core reasoning engine governed by Topo-CBP immutable bedrock clamping.\n"
            f"User: Original Query: {initial_query}\n\nDeconstructed Subtasks:\n{subtasks}\n\n"
            f"Execute rigorous step-by-step reasoning and deliver the final verified solution.\n"
            f"Assistant:"
        )

        r1_response = self.r1_engine(
            r1_prompt,
            max_tokens=2048,
            temperature=0.6,
            stop=["User:", "System:"]
        )
        final_reasoning = r1_response["choices"][0]["text"].strip()
        print(f"[+] R1-70B Deep Reasoning complete.")

        return {
            "subtasks": subtasks,
            "final_solution": final_reasoning
        }

if __name__ == "__main__":
    MOE_PATH = "/content/drive/MyDrive/GGUF_Models/deepseek_moe_16b_topocbp_q4_k_m.gguf"
    R1_70B_PATH = "/content/drive/MyDrive/GGUF_Models/deepseek-r1-distill-llama-70b-topo-governed-q8_0.gguf"

    orchestrator = DualModelSovereignOrchestrator(
        moe_gguf_path=MOE_PATH,
        r1_70b_gguf_path=R1_70B_PATH
    )

    query = "Formulate a zero-drift continuous-learning update rule integrating the Laplace-Euler-Fourier-Mellin operator."

    result = orchestrator.run_agentic_workflow(query)

    print("\n==============================")
    print("FINAL DUAL-MODEL AGENTIC OUTPUT")
    print("==============================")
    print(result["final_solution"])

[*] Initializing SOMALA Dual-Model Agentic Orchestrator from Google Drive...
[*] Loading MoE from: /content/drive/MyDrive/GGUF_Models/deepseek_moe_16b_topocbp_q4_k_m.gguf
[*] Loading R1-70B from: /content/drive/MyDrive/GGUF_Models/deepseek-r1-distill-llama-70b-topo-governed-q8_0.gguf
[+] Dual-Model Bare-Metal Engine Active under Topo-CBP Governance.

[Step 1] Dispatching query to DeepSeek-MoE-16B for task decomposition...
[+] MoE Decomposition complete:
You are the MoE routing and task decomposition worker governed by Topo-CBP. Deconstruct incoming queries into concise analytical subtasks.

[Step 2] Passing subtasks to DeepSeek-R1-Distill-Llama-70B for zero-drift reasoning...
[+] R1-70B Deep Reasoning complete.

FINAL DUAL-MODEL AGENTIC OUTPUT
This is a complex mathematical problem. Let me try to break it down step by step. First, I need to understand what a zero-drift continuous learning update rule is. Continuous learning typically refers to a system that can learn incrementally from

## case2

In [1]:
from llama_cpp import Llama

class MedicalDualModelOrchestrator:
    def __init__(self, moe_gguf_path: str, r1_70b_gguf_path: str):
        print("[*] Initializing Medical SOMALA Orchestrator...")

        # Model 1: Structural Text Extraction & Entity Parsing
        print(f"[*] Loading MoE Triage Engine from: {moe_gguf_path}")
        self.moe_engine = Llama(
            model_path=moe_gguf_path,
            n_ctx=4096,
            n_threads=4,
            n_gpu_layers=-1,
            verbose=False
        )

        # Model 2: Deep Clinical Reasoning & Differential Diagnosis
        print(f"[*] Loading R1-70B Clinical Reasoner from: {r1_70b_gguf_path}")
        self.r1_engine = Llama(
            model_path=r1_70b_gguf_path,
            n_ctx=8192,
            n_threads=8,
            n_gpu_layers=-1,
            verbose=False
        )
        print("[+] Medical Dual-Model Engine Active.")

    def run_clinical_workflow(self, case_narrative: str) -> dict:
        print(f"\n[Step 1] Dispatching case to DeepSeek-MoE-16B for structural entity extraction...")

        # Reframed prompt to avoid MoE safety refusal triggers
        moe_prompt = (
            f"System: You are a text parsing and categorization assistant. "
            f"Extract key descriptive terms, physical characteristics, and technical measurements into bullet points.\n"
            f"User: Extract key entities from this text: {case_narrative}\n"
            f"Assistant:"
        )

        moe_response = self.moe_engine(
            moe_prompt,
            max_tokens=512,
            temperature=0.2,
            stop=["User:", "System:", "\n\n\n"]
        )
        extracted_features = moe_response["choices"][0]["text"].strip()
        print(f"[+] Feature Extraction complete:\n{extracted_features}")

        print(f"\n[Step 2] Passing structured features to DeepSeek-R1-Distill-Llama-70B for differential diagnosis...")

        r1_prompt = (
            f"System: You are an expert academic neurologist and neuroradiologist. Provide a rigorous differential diagnosis, "
            f"addressing anatomical correlates, neuroimaging hallmarks (e.g., midbrain atrophy signs), and recommended diagnostic next steps.\n"
            f"User: Original Case: {case_narrative}\n\nExtracted Features:\n{extracted_features}\n\n"
            f"Perform differential diagnosis and establish a clinical validation plan.\n"
            f"Assistant:"
        )

        r1_response = self.r1_engine(
            r1_prompt,
            max_tokens=2048,
            temperature=0.5,
            stop=["User:", "System:"]
        )
        clinical_assessment = r1_response["choices"][0]["text"].strip()
        print(f"[+] R1-70B Clinical Reasoning complete.")

        return {
            "extracted_features": extracted_features,
            "clinical_assessment": clinical_assessment
        }

if __name__ == "__main__":
    MOE_PATH = "/content/drive/MyDrive/GGUF_Models/deepseek_moe_16b_topocbp_q4_k_m.gguf"
    R1_70B_PATH = "/content/drive/MyDrive/GGUF_Models/deepseek-r1-distill-llama-70b-topo-governed-q8_0.gguf"

    orchestrator = MedicalDualModelOrchestrator(
        moe_gguf_path=MOE_PATH,
        r1_70b_gguf_path=R1_70B_PATH
    )

    case = (
        "58-year-old patient presents with progressive asymmetric rigidity, resting tremor, early postural instability, "
        "and subtle executive dysfunction over 14 months, unresponsive to high-dose levodopa. "
        "MRI brain demonstrates midbrain atrophy with hummingbird sign and putaminal hypointensity."
    )

    result = orchestrator.run_clinical_workflow(case)

    print("\n==============================")
    print("FINAL CLINICAL ASSESSMENT REPORT")
    print("==============================")
    print(result["clinical_assessment"])

[*] Initializing Medical SOMALA Orchestrator...
[*] Loading MoE Triage Engine from: /content/drive/MyDrive/GGUF_Models/deepseek_moe_16b_topocbp_q4_k_m.gguf
[*] Loading R1-70B Clinical Reasoner from: /content/drive/MyDrive/GGUF_Models/deepseek-r1-distill-llama-70b-topo-governed-q8_0.gguf
[+] Medical Dual-Model Engine Active.

[Step 1] Dispatching case to DeepSeek-MoE-16B for structural entity extraction...
[+] Feature Extraction complete:
58-year-old patient presents with progressive asymmetric rigidity, resting tremor, early postural instability, and subtle executive dysfunction over 14 months, unresponsive to high-dose levodopa. MRI brain demonstrates midbrain atrophy with hummingbird sign and putaminal hypointensity.

[Step 2] Passing structured features to DeepSeek-R1-Distill-Llama-70B for differential diagnosis...
[+] R1-70B Clinical Reasoning complete.

FINAL CLINICAL ASSESSMENT REPORT
Differential Diagnosis and Clinical Validation Plan for a 58-Year-Old Patient with Progressive A

In [2]:
!nvidia-smi

Thu Nov 12 14:10:38 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA RTX PRO 6000 Blac...    Off |   00000000:05:00.0 Off |                    0 |
| N/A   34C    P0             88W /  600W |   86227MiB /  97887MiB |      0%      Default |
|                                         |                        |             Disabled |
+-----------------------------------------+-----